In [ ]:
import os
os.listdir("/kaggle/input/datasets/rutujapadmakarlahane/earthformer-repo")

import sys

sys.path.append("/kaggle/input/datasets/rutujapadmakarlahane/earthformer-repo/src")

from earthformer.cuboid_transformer.cuboid_transformer import CuboidTransformerModel

print("Contents of /kaggle/input:")
for name in os.listdir("/kaggle/input/"):
    print(" -", name)

In [ ]:
# ============================================================
# EARTHFORMER (Dynamic)
# ============================================================

import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from earthformer.cuboid_transformer.cuboid_transformer import CuboidTransformerModel
import random

# -------------------------
# SEED FIXING
# -------------------------
SEED = 45
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# CONFIG
# -------------------------
DATA_DIR = r"/kaggle/input/datasets/rutujapadmakarlahane/earthformer-arrays-dynamic/Earthformer_Arrays_Dynamic"
CYCLONES = ["Amphan","Dana","Fani","Gulab","Hudhud","Phailin","Titli","Yaas"]
TRAIN_CYCLONES = ["Amphan","Dana","Fani","Gulab","Hudhud","Phailin","Titli"]
VAL_CYCLONES = ["Asani"]
TEST_CYCLONE = ["Yaas"]

BATCH_SIZE = 1
EPOCHS = 20
LR = 1e-4
VARS = ["to","so","ugo","vgo","zo"]
VARS_OUTPUT = ["to","so","zo"]

# -------------------------
# DATASET
# -------------------------
class CycloneDepthDataset(Dataset):
    """Depth-inclusive cyclone dataset (X without forcing, Y with forcing)"""
    def __init__(self, names):
        self.samples = []
        for n in names:
            X = np.load(os.path.join(DATA_DIR, f"{n}_X.npy"))  # (T,H,W,Cin)
            Y = np.load(os.path.join(DATA_DIR, f"{n}_Y.npy"))  # (T,H,W,Cout)
            self.samples.append((X, Y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        X, Y = self.samples[idx]
        return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)

# -------------------------
# TRAIN / VAL SPLIT
# -------------------------
val_loader = DataLoader(CycloneDepthDataset(VAL_CYCLONES), batch_size=1, shuffle=False)
train_loader = DataLoader(CycloneDepthDataset(TRAIN_CYCLONES), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(CycloneDepthDataset(TEST_CYCLONE), batch_size=1, shuffle=False)

# -------------------------
# INFER SHAPES
# -------------------------
X0, Y0 = next(iter(train_loader))
_, Tin, H, W, Cin = X0.shape
Tout = Y0.shape[1]
Cout = Y0.shape[-1]

print(f"Inferred Shapes → Tin: {Tin}, Tout: {Tout}, H: {H}, W: {W}, Cin: {Cin}, Cout: {Cout}")

# -------------------------
# EARLY STOPPING
# -------------------------
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

# -------------------------
# MODEL INITIALIZATION
# -------------------------
model = CuboidTransformerModel(
    input_shape=(Tin, H, W, Cin),
    target_shape=(Tout, H, W, Cout),
    enc_cuboid_size=[(2,2,2),(2,2,2),(2,2,2)],
    enc_cuboid_strategy=[('l','l','l'),('d','d','d'),('d','l','l')],
    enc_shift_size=[(0,0,0),(1,1,1),(0,1,1)],
    base_units=128,
    enc_depth=[4,4,4],
    dec_depth=[4,4,4],
    num_heads=4,
    block_units=None,
    scale_alpha=1.0,
    attn_drop=0.0,
    proj_drop=0.0,
    ffn_drop=0.0,
    downsample=2,
    downsample_type='patch_merge',
    upsample_type='upsample',
    upsample_kernel_size=3,
).to(device)

# -------------------------
# LOSS & OPTIMIZER
# -------------------------
criterion = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

MODEL_SAVE_PATH = "/kaggle/working/Earthformer_Model_Dynamic_2.pt"
early_stopper = EarlyStopping(patience=5)

best_val_loss = np.inf
best_epoch = -1

train_losses = []
val_losses = []

for epoch in range(EPOCHS):

    # ------------------
    # TRAIN
    # ------------------
    model.train()
    train_loss = 0.0

    for X, Y in train_loader:
        X, Y = X.to(device), Y.to(device)
        optimizer.zero_grad()
        Y_hat = model(X)
        loss = criterion(Y_hat, Y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ------------------
    # VALIDATION
    # ------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X, Y in val_loader:
            X, Y = X.to(device), Y.to(device)
            Y_hat = model(X)
            loss = criterion(Y_hat, Y)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1:03d} | Train MSE: {train_loss:.4e} | Val MSE: {val_loss:.4e}")

    # Early stopping on VAL loss
    early_stopper(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print("✅ Best model saved.")

    if early_stopper.early_stop:
        print(f"Stopping early at epoch {epoch+1}")
        break



model.load_state_dict(torch.load(MODEL_SAVE_PATH))
print("Loaded best validation model for testing.")

import matplotlib.pyplot as plt

plt.figure()
plt.plot(train_losses)
plt.plot(val_losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training vs Validation Loss")
plt.legend(["Train Loss", "Validation Loss"])
plt.savefig("/kaggle/working/training_validation_curve_dynamic_2.png")
plt.show()

# -------------------------
# TEST
# -------------------------
model.eval()
all_Y_true, all_Y_pred = [], []

with torch.no_grad():
    for X, Y in test_loader:
        X, Y = X.to(device), Y.to(device)
        Y_hat = model(X)
        all_Y_true.append(Y.cpu().numpy())
        all_Y_pred.append(Y_hat.cpu().numpy())

Y_true = np.concatenate(all_Y_true, axis=0)
Y_pred = np.concatenate(all_Y_pred, axis=0)

PRED_SAVE_DIR = "/kaggle/working/Test_Predictions_Dynamic_2"
os.makedirs(PRED_SAVE_DIR, exist_ok=True)
np.save(os.path.join(PRED_SAVE_DIR,"Y_true.npy"),Y_true)
np.save(os.path.join(PRED_SAVE_DIR,"Y_pred.npy"),Y_pred)
print(f"\n✅ Test predictions saved at: {PRED_SAVE_DIR}")

# -------------------------
# METRICS (Total)
# -------------------------
mse = np.mean((Y_pred - Y_true)**2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(Y_pred - Y_true))
r2 = 1 - np.sum((Y_true - Y_pred)**2) / np.sum((Y_true - Y_true.mean())**2)

print("\nFinal Test Metrics")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")

# -------------------------
# METRICS (only physical variables, exclude depth + forcing)
# -------------------------
var_channels = slice(0, len(VARS_OUTPUT))  # only the 5 physical variables
Y_true_sel = Y_true[..., var_channels]
Y_pred_sel = Y_pred[..., var_channels]

mse = np.mean((Y_pred_sel - Y_true_sel)**2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(Y_pred_sel - Y_true_sel))
r2 = 1 - np.sum((Y_true_sel - Y_pred_sel)**2) / np.sum((Y_true_sel - Y_true_sel.mean())**2)

print("\nFinal Test Metrics (physical variables only)")
print(f"MSE: {mse:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")